<a href="https://colab.research.google.com/github/yaroslavkarpenko701/Ecommerce-ab-testing-analysis/blob/main/Portfolio_Project_2_(A_B_Test).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


In [ ]:
%cd /content/drive/MyDrive/online_store_sales_analytics/portfolio_project_2_AB_Test

/content/drive/MyDrive/online_store_sales_analytics/portfolio_project_2_AB_Test


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

import statsmodels.api as sm
import scipy.stats as scst

In [ ]:
df = pd.read_csv("create_your_AB_testing_tool.csv")

Отримали дані з csv файлу(`create_your_AB_testing_tool.csv`), з гугл диску

In [ ]:
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-02,Tunisia,desktop,Africa,Organic Search,2,1,new account,1
1,2020-11-04,Jordan,desktop,Asia,Social Search,2,1,new account,1
2,2020-11-05,New Zealand,desktop,Oceania,Undefined,2,1,new account,1
3,2020-11-06,Serbia,desktop,Europe,Organic Search,2,1,new account,1
4,2020-11-06,New Zealand,desktop,Oceania,Paid Search,2,1,new account,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  object
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  int64 
 6   test_group  800996 non-null  int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 55.0+ MB


In [ ]:
df.isna().sum()

,0
date,0
country,0
device,0
continent,0
channel,0
test,0
test_group,0
event_name,0
value,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df["date"] = pd.to_datetime(df["date"])

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        800996 non-null  datetime64[ns]
 1   country     800996 non-null  object        
 2   device      800996 non-null  object        
 3   continent   800996 non-null  object        
 4   channel     800996 non-null  object        
 5   test        800996 non-null  int64         
 6   test_group  800996 non-null  int64         
 7   event_name  800996 non-null  object        
 8   value       800996 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(5)
memory usage: 55.0+ MB


In [ ]:
for col in df.columns:
  print(f"{col}:\n {df[col].unique()}\n")

date:
 <DatetimeArray>
['2020-11-02 00:00:00', '2020-11-04 00:00:00', '2020-11-05 00:00:00',
 '2020-11-06 00:00:00', '2020-11-07 00:00:00', '2020-11-09 00:00:00',
 '2020-11-10 00:00:00', '2020-11-11 00:00:00', '2020-11-13 00:00:00',
 '2020-11-14 00:00:00', '2020-11-15 00:00:00', '2020-11-16 00:00:00',
 '2020-11-20 00:00:00', '2020-11-21 00:00:00', '2020-11-24 00:00:00',
 '2020-11-25 00:00:00', '2020-11-28 00:00:00', '2020-11-19 00:00:00',
 '2020-11-29 00:00:00', '2020-11-03 00:00:00', '2020-11-17 00:00:00',
 '2020-11-18 00:00:00', '2020-11-23 00:00:00', '2020-11-26 00:00:00',
 '2020-12-24 00:00:00', '2021-01-01 00:00:00', '2021-01-05 00:00:00',
 '2021-01-06 00:00:00', '2021-01-07 00:00:00', '2021-01-10 00:00:00',
 '2021-01-12 00:00:00', '2021-01-17 00:00:00', '2021-01-18 00:00:00',
 '2021-01-23 00:00:00', '2020-12-21 00:00:00', '2020-12-26 00:00:00',
 '2020-12-29 00:00:00', '2021-01-08 00:00:00', '2021-01-14 00:00:00',
 '2020-12-07 00:00:00', '2020-12-10 00:00:00', '2020-12-19 00:00:00

**device:**
 ['desktop' 'mobile' 'tablet']

**continent:**
 ['Africa' 'Asia' 'Oceania' 'Europe' 'Americas' '(not set)']

**channel:**
 ['Organic Search' 'Social Search' 'Undefined' 'Paid Search' 'Direct']

**test:**
 [2 1 4 3]

**test_group:**
 [1 2]

**event_name:**
 ['new account' 'session' 'session with orders' 'page_view' 'scroll'
 'user_engagement' 'view_promotion' 'session_start' 'first_visit'
 'view_item' 'view_search_results' 'select_item' 'begin_checkout'
 'add_to_cart' 'add_shipping_info' 'click' 'add_payment_info'
 'select_promotion' 'view_item_list']

In [ ]:
df_copy = df.copy()

In [ ]:
df_copy["event_name"] = df_copy["event_name"].str.replace(" ", "_")

Виконано змінення вигляду деяких назв **event_name**, щоб всі мали один тип оформлення, ***snake_case***

Перевірили всі дані, змінили тип даних для дати, та виправлено назви подій

In [ ]:
for col in df_copy.columns:
  print(f"{col}:\n {df_copy[col].unique()}\n")

date:
 <DatetimeArray>
['2020-11-02 00:00:00', '2020-11-04 00:00:00', '2020-11-05 00:00:00',
 '2020-11-06 00:00:00', '2020-11-07 00:00:00', '2020-11-09 00:00:00',
 '2020-11-10 00:00:00', '2020-11-11 00:00:00', '2020-11-13 00:00:00',
 '2020-11-14 00:00:00', '2020-11-15 00:00:00', '2020-11-16 00:00:00',
 '2020-11-20 00:00:00', '2020-11-21 00:00:00', '2020-11-24 00:00:00',
 '2020-11-25 00:00:00', '2020-11-28 00:00:00', '2020-11-19 00:00:00',
 '2020-11-29 00:00:00', '2020-11-03 00:00:00', '2020-11-17 00:00:00',
 '2020-11-18 00:00:00', '2020-11-23 00:00:00', '2020-11-26 00:00:00',
 '2020-12-24 00:00:00', '2021-01-01 00:00:00', '2021-01-05 00:00:00',
 '2021-01-06 00:00:00', '2021-01-07 00:00:00', '2021-01-10 00:00:00',
 '2021-01-12 00:00:00', '2021-01-17 00:00:00', '2021-01-18 00:00:00',
 '2021-01-23 00:00:00', '2020-12-21 00:00:00', '2020-12-26 00:00:00',
 '2020-12-29 00:00:00', '2021-01-08 00:00:00', '2021-01-14 00:00:00',
 '2020-12-07 00:00:00', '2020-12-10 00:00:00', '2020-12-19 00:00:00

In [ ]:
# Створюємо словник з метриками для тестів

metrics = {
    "add_payment_info": "session",
    "add_shipping_info": "session",
    "begin_checkout": "session",
    "new_account": "session",
    "select_item": "session"
}

# 1) Основний розрахунок A/B тесту, по тоталу для кожного тесту

In [ ]:
# Зведена таблиця для роботи з заданими метриками та розрізами

df_pivot = df_copy.pivot_table(
    index = ["test", "test_group"],
    columns = "event_name",
    values = "value",
    aggfunc = "sum"
).reset_index()

df_pivot.head(10)

event_name,test,test_group,add_payment_info,add_shipping_info,add_to_cart,begin_checkout,click,first_visit,new_account,page_view,...,select_item,select_promotion,session,session_start,session_with_orders,user_engagement,view_item,view_item_list,view_promotion,view_search_results
0,1,1,1988,3034,1395,3784,368,30596,3823,191543,...,543,1275,45362,45905,4514,171788,62335,27,29188,3678
1,1,2,2229,3221,1366,4021,353,30512,3681,198050,...,530,1323,45193,45649,4526,179081,65337,24,29117,3882
2,2,1,2344,3480,2811,4262,337,34511,4165,220275,...,905,1477,50637,51219,5102,198266,72717,24,32367,4282
3,2,2,2409,3510,3061,4313,413,34171,4184,212320,...,946,1406,50244,50808,5003,189931,68700,29,31680,4198
4,3,1,3623,5298,17674,9532,280,50438,5856,286351,...,8735,2020,70047,71312,6951,249921,93931,9,41169,5764
5,3,2,3697,5188,17229,9264,292,50728,5822,284287,...,8556,1996,70439,71686,7151,248325,92841,9,40959,5619
6,4,1,3731,5128,21536,12555,285,80900,8984,379480,...,12214,2773,105079,107128,10596,273946,98619,5,52672,7582
7,4,2,3601,4956,21554,12267,299,80626,8687,378307,...,11988,2708,105141,106978,10481,273633,98928,6,51985,7203


In [ ]:
df_pivot.isna().sum()

,0
event_name,
test,0
test_group,0
add_payment_info,0
add_shipping_info,0
add_to_cart,0
begin_checkout,0
click,0
first_visit,0
new_account,0


In [ ]:
results = []


for test_number in df_pivot["test"].unique(): # перший цикл по номеру тесту
  df_filter = df_pivot[df_pivot["test"] == test_number] # обираємо варіант тестування

  # робимо дві серії для 1 та 2 групи
  control_row = df_filter[df_filter["test_group"] == 1].iloc[0]
  experimental_row = df_filter[df_filter["test_group"] == 2].iloc[0]

  for numerator_event, denominator_event in metrics.items(): # другий цикл по метрикам з словника
    metric_name = f"{numerator_event} / {denominator_event}"

    # отримуємо значення чисельника та знаменика для 1 групи
    numerator_control = control_row[numerator_event]
    denominator_control = control_row[denominator_event]

    # отримуємо значення чисельника та знаменика для 2 групи
    numerator_experimental = experimental_row[numerator_event]
    denominator_experimental = experimental_row[denominator_event]

    # отримуємо конверсії для обох груп
    conversion_control = (numerator_control / denominator_control) * 100 if denominator_control > 0 else 0
    conversion_experimental = (numerator_experimental / denominator_experimental) * 100 if denominator_experimental > 0 else 0

    # отримуємо відносну зміну між конверсіями
    if conversion_control > 0:
      metric_change = (conversion_experimental - conversion_control) / conversion_control * 100
    else:
      metric_change = 0

    # отримуємо результати тесту пропорцій
    z_stat, p_value = sm.stats.proportions_ztest([numerator_control, numerator_experimental],[denominator_control, denominator_experimental])

    # перевіряємо на ститисничну значущість результат тесту пропорцій
    significant = p_value < 0.05

    results.append({
            "test_number": test_number,
            "metric": metric_name,
            "numerator_event": numerator_event,
            "denominator_event": denominator_event,
            "numerator_control": numerator_control,
            "denominator_control": denominator_control,
            "conversion_rate_control,%": conversion_control,
            "numerator_experimental": numerator_experimental,
            "denominator_experimental": denominator_experimental,
            "conversion_rate_experimental,%": conversion_experimental,
            "metric_change,%": metric_change,
            "z_stat": z_stat,
            "p_value": p_value,
            "significant": significant
        })

df_results = pd.DataFrame(results)

df_results

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,"conversion_rate_control,%",numerator_experimental,denominator_experimental,"conversion_rate_experimental,%","metric_change,%",z_stat,p_value,significant
0,1,add_payment_info / session,add_payment_info,session,1988,45362,4.382523,2229,45193,4.932180,12.542021,-3.924884,0.000087,True
1,1,add_shipping_info / session,add_shipping_info,session,3034,45362,6.688418,3221,45193,7.127210,6.560481,-2.603571,0.009226,True
2,1,begin_checkout / session,begin_checkout,session,3784,45362,8.341784,4021,45193,8.897396,6.660587,-2.978783,0.002894,True
3,1,new_account / session,new_account,session,3823,45362,8.427759,3681,45193,8.145067,-3.354299,1.542883,0.122859,False
4,1,select_item / session,select_item,session,543,45362,1.197037,530,45193,1.172748,-2.029108,0.337740,0.735559,False
5,2,add_payment_info / session,add_payment_info,session,2344,50637,4.629026,2409,50244,4.794602,3.576911,-1.240994,0.214608,False
6,2,add_shipping_info / session,add_shipping_info,session,3480,50637,6.872445,3510,50244,6.985909,1.650995,-0.709557,0.477979,False
7,2,begin_checkout / session,begin_checkout,session,4262,50637,8.416770,4313,50244,8.584110,1.988164,-0.952898,0.340642,False
8,2,new_account / session,new_account,session,4165,50637,8.225211,4184,50244,8.327362,1.241934,-0.588793,0.556000,False
9,2,select_item / session,select_item,session,905,50637,1.787231,946,50244,1.882812,5.348006,-1.131011,0.258051,False


In [ ]:
df_results.to_csv("ab_test_significance_results_total.csv", index = False, sep = ";")

Для 4 тестів, 2 тестових груп та 5 метрик, маємо:

* З 20 можливих комбінацій статистично значущу різницю зафіксовано у 6 випадках.
* У 9 випадках спостерігалося позитивне змінення метрики, у 11 — негативне.
* Найбільш стабільно значущий позитивний ефект спостерігався в Тесті 1 (майже всі метрики показали значуще зростання).
* У Тесті 2 жодна з метрик не досягла рівня значущості.
* У Тестах 3 та 4 значущі зміни переважно негативні.

Потрібно дослідити дані результати в розрізі різних сегментів, щоб побачити ситуацію більш детально.

# 2) Розрахуноки A/B тестування з більшою кількістю розрізів

In [ ]:
metrics = {
    "add_payment_info": "session",
    "add_shipping_info": "session",
    "begin_checkout": "session",
    "new_account": "session",
    "select_item": "session"
}

In [ ]:
# список з розрізами які необхідно виконати

dims = ["test", "continent", "country", "device", "channel"]
#dims = ["test", "continent", "country"]
#dims = ["test"]

group_cols = dims + ["test_group"]

# Зведена таблиця для роботи з заданими метриками та розрізами
df_pivot_2 = df_copy.pivot_table(
    index = group_cols,
    columns = "event_name",
    values = "value",
    aggfunc = "sum",
    fill_value = 0
).reset_index()

In [ ]:
print(group_cols)


['test', 'continent', 'country', 'device', 'channel', 'test_group']


In [ ]:
df_pivot_2[group_cols + ["session", "add_payment_info"]].head()

event_name,test,continent,country,device,channel,test_group,session,add_payment_info
0,1,(not set),(not set),desktop,Direct,1,15,0
1,1,(not set),(not set),desktop,Direct,2,14,0
2,1,(not set),(not set),desktop,Organic Search,1,19,0
3,1,(not set),(not set),desktop,Organic Search,2,29,3
4,1,(not set),(not set),desktop,Paid Search,1,8,0


In [ ]:
print(df_pivot_2["session"].sum())

542142


In [ ]:
print(df_pivot["session"].sum())

542142


In [ ]:
results_2 = []

# Перебираємо комбінації вибраних вимірів
for segment_values, segment_df in df_pivot_2.groupby(dims):

    # Знаходимо контрольну групу
    control_row = segment_df[segment_df["test_group"] == 1]

    # Знаходимо експериментальну групу
    experimental_row = segment_df[segment_df["test_group"] == 2]

    # Перевіряємо, чи знайшли по одному рядку для кожної групи, якщо ні то припускаємо цей рядок
    if len(control_row) != 1 or len(experimental_row) != 1:
        continue

    # Перетворюємо однорядкові таблиці на окремі рядки Series
    control_row = control_row.iloc[0]
    experimental_row = experimental_row.iloc[0]

    # Перебираємо всі метрики
    for numerator_event, denominator_event in metrics.items():
        metric_name = f"{numerator_event} / {denominator_event}"


        # отримуємо значення чисельника та знаменика для 1 групи
        numerator_control = control_row[numerator_event]
        denominator_control = control_row[denominator_event]

        # отримуємо значення чисельника та знаменика для 2 групи
        numerator_experimental = experimental_row[numerator_event]
        denominator_experimental = experimental_row[denominator_event]

        # Рахуємо конверсії
        conversion_control = (numerator_control / denominator_control) * 100 if denominator_control > 0 else 0
        conversion_experimental = (numerator_experimental / denominator_experimental) * 100 if denominator_experimental > 0 else 0

        # отримуємо відносну зміну між конверсіями
        if conversion_control > 0:
          metric_change = (conversion_experimental - conversion_control) / conversion_control * 100
        else:
          metric_change = 0

        # Загальна кількість подій у двох групах
        total_successes = numerator_control + numerator_experimental

        # Загальна кількість сесій у двох групах
        total_trials = denominator_control + denominator_experimental

        # Перевіряємо, чи коректні дані для тесту та отримуємо результати тесту пропорцій
        if (
          denominator_control <= 0
          or denominator_experimental <= 0
          or numerator_control < 0
          or numerator_experimental < 0
          or numerator_control > denominator_control
          or numerator_experimental > denominator_experimental
          ):
            z_stat = np.nan
            p_value = np.nan
            significant = False

        # перевірка на нуль подій в обох групах або 100% конверсія в обох групах
        elif total_successes == 0 or total_successes == total_trials:
          z_stat = np.nan
          p_value = np.nan
          significant = False

        else:
          z_stat, p_value = sm.stats.proportions_ztest([numerator_control, numerator_experimental],[denominator_control, denominator_experimental])

          # перевіряємо на ститисничну значущість результат тесту пропорцій
          significant = p_value < 0.05

        segment_info = dict(zip(dims, segment_values)) # словник в якому поєднуємо кожну назву колонки з її значенням

        results_2.append({
            **segment_info, # всі пари з словника
            "metric": metric_name,
            "numerator_event": numerator_event,
            "denominator_event": denominator_event,
            "numerator_control": numerator_control,
            "denominator_control": denominator_control,
            "conversion_rate_control,%": conversion_control,
            "numerator_experimental": numerator_experimental,
            "denominator_experimental": denominator_experimental,
            "conversion_rate_experimental,%": conversion_experimental,
            "metric_change,%": metric_change,
            "z_stat": z_stat,
            "p_value": p_value,
            "significant": significant
        })

df_results_2 = pd.DataFrame(results_2)

df_results_2






,test,continent,country,device,channel,metric,numerator_event,denominator_event,numerator_control,denominator_control,"conversion_rate_control,%",numerator_experimental,denominator_experimental,"conversion_rate_experimental,%","metric_change,%",z_stat,p_value,significant
0,1,(not set),(not set),desktop,Direct,add_payment_info / session,add_payment_info,session,0,15,0.0,0,14,0.000000,0.000000,NaN,NaN,False
1,1,(not set),(not set),desktop,Direct,add_shipping_info / session,add_shipping_info,session,0,15,0.0,1,14,7.142857,0.000000,-1.053420,0.292149,False
2,1,(not set),(not set),desktop,Direct,begin_checkout / session,begin_checkout,session,0,15,0.0,2,14,14.285714,0.000000,-1.517098,0.129242,False
3,1,(not set),(not set),desktop,Direct,new_account / session,new_account,session,3,15,20.0,1,14,7.142857,-64.285714,1.003352,0.315691,False
4,1,(not set),(not set),desktop,Direct,select_item / session,select_item,session,0,15,0.0,0,14,0.000000,0.000000,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21795,4,Oceania,New Zealand,mobile,Undefined,add_payment_info / session,add_payment_info,session,0,6,0.0,0,2,0.000000,0.000000,NaN,NaN,False
21796,4,Oceania,New Zealand,mobile,Undefined,add_shipping_info / session,add_shipping_info,session,0,6,0.0,0,2,0.000000,0.000000,NaN,NaN,False
21797,4,Oceania,New Zealand,mobile,Undefined,begin_checkout / session,begin_checkout,session,0,6,0.0,0,2,0.000000,0.000000,NaN,NaN,False
21798,4,Oceania,New Zealand,mobile,Undefined,new_account / session,new_account,session,0,6,0.0,0,2,0.000000,0.000000,NaN,NaN,False


In [ ]:
check = df_results_2[(df_results_2["test"] == 2) & (df_results_2["metric"] == "add_payment_info / session")]

In [ ]:
check["numerator_control"].sum()

np.int64(2330)

Під час збільшення кількості вимірів для сегментації даних кількість комбінацій зростає. Наприклад, аналіз лише за номером тесту охоплює більше спостережень, ніж аналіз за тестом, континентом, країною, пристроєм і каналом одночасно.

Для проведення тесту пропрцій, в кожному сегменті, мають бути представлені як контрольна, так і експериментальна групи. Якщо в певній комбінації одна з груп відсутня, цей сегмент пропускається, оскільки порівняти конверсії двох груп у ньому неможливо.

Наприклад, для тесту `№ 2` та метрики `add_payment_info / session` у контрольній групі становить `2344` під час аналізу лише за номером тесту. Після додавання вимірів continent, country, device і channel сума подій у сегментах, які пройшли перевірку, становить `2330`.

Різниця виникає через те, що частина подій належить сегментам, у яких відсутня одна з тестових груп. Тому сума подій у сегментах, що пройшли перевірку, може бути меншою за загальну кількість подій у тесті.


In [ ]:
df_results_2.to_csv("ab_test_significance_results.csv", index = False, sep = ";")

* Tableau:

    https://public.tableau.com/app/profile/yaroslav.karpenko/viz/ABtest1_17917144023580/ABtest?publish=yes

* CSV файл з розрізом тільки по тесту:

    https://drive.google.com/file/d/1d9_CTOmn2UVc4KgwseplKbWv7Mz4JUnT/view?usp=sharing


* CSV файл з розрізом по різним категоріям:

    https://drive.google.com/file/d/12hCZMe_8EwAl5u3GvcTk3GTXfYV0HhJU/view?usp=sharing
